<a href="https://colab.research.google.com/github/2303A51908/Reinforecement-Learning---B12/blob/main/2303A51908_RL_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gymnasium as gym
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

env = gym.make("CartPole-v1")

def expert_policy(obs):
    # Simple heuristic: push in direction of pole angle
    angle = obs[2]
    return 1 if angle > 0 else 0

expert_obs = []
expert_actions = []

def collect_expert_data(n_episodes=20):
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action = expert_policy(obs)

            expert_obs.append(obs)
            expert_actions.append(action)

            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

collect_expert_data(n_episodes=20)
expert_obs = np.array(expert_obs, dtype=np.float32)
expert_actions = np.array(expert_actions, dtype=np.int64)

print("Collected samples:", len(expert_obs))


Collected samples: 851


In [ ]:
class ExpertDataset(Dataset):
    def __init__(self, observations, actions):
        self.obs = torch.tensor(observations, dtype=torch.float32)
        self.actions = torch.tensor(actions, dtype=torch.long)

    def __len__(self):
        return len(self.obs)

    def __getitem__(self, idx):
        return self.obs[idx], self.actions[idx]

dataset = ExpertDataset(expert_obs, expert_actions)
loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [ ]:
import torch.nn as nn

class BCPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)   # CartPole actions: 0 or 1
        )

    def forward(self, x):
        return self.net(x)

model = BCPolicy()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


In [ ]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    for obs_batch, act_batch in loader:
        logits = model(obs_batch)
        loss = criterion(logits, act_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f}")


Epoch 1/10 - Loss: 9.6237
Epoch 2/10 - Loss: 9.4171
Epoch 3/10 - Loss: 9.1521
Epoch 4/10 - Loss: 8.7714
Epoch 5/10 - Loss: 8.1932
Epoch 6/10 - Loss: 7.4148
Epoch 7/10 - Loss: 6.5000
Epoch 8/10 - Loss: 5.6145
Epoch 9/10 - Loss: 4.7538
Epoch 10/10 - Loss: 4.1174


In [ ]:
def evaluate(model, episodes=5):
    model.eval()
    total_reward = 0

    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = 0

        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32)
            action = torch.argmax(model(obs_tensor)).item()

            obs, reward, terminated, truncated, _ = env.step(action)
            ep_reward += reward
            done = terminated or truncated

        total_reward += ep_reward
        print("Episode reward:", ep_reward)

    print("Average reward:", total_reward / episodes)

evaluate(model)


Episode reward: 34.0
Episode reward: 38.0
Episode reward: 40.0
Episode reward: 38.0
Episode reward: 39.0
Average reward: 37.8
